# 1. Dataset

In [79]:
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2
from torch import randint
import os
import random
from torchvision import transforms
import pandas as pd
from PIL import Image


def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


mean = [0.66861665, 0.4143819,  0.2288029]
std = [0.14154758, 0.10918795, 0.07485254]
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.ColorJitter(), ]), p=0.3),
        transforms.RandomApply(torch.nn.ModuleList([transforms.GaussianBlur(kernel_size=3), ]), p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

class PapilledemaDataset(Dataset): 
    def __init__(self, 
                data_path = "../VinDr_Mammo/physionet.org/files/vindr-mammo/1.0.0/images_png/",
                phase ='train',
                transform=None,
                seed=None):
        self.phase = phase
        self.data_path= os.path.join(data_path, self.phase)
        self.transform = data_transforms[self.phase] if (transform == None) else transform
        if(seed):
            seed_everything(seed)

        self.image_path_list = []
        self.label_list = []

        for label in ["Normal", "Pseudopapilledema", "Papilledema"]:
            label_image_folder_path = os.path.join(self.data_path, label)
            
            for image in os.listdir(label_image_folder_path):
                image_path = os.path.join(label_image_folder_path, image)
                self.image_path_list.append(image_path)
                self.label_list.append(0 if label == "Normal" else 1 if label == "Pseudopapilledema" else 0)
        
    
    def __getitem__(self, index):
        image_path = self.image_path_list[index]
        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.label_list[index])
        return image, label 
    
    
    def __len__(self):
        return len(self.image_path_list)

# 2. Base model

In [80]:
from torch import nn
def get_default_fc(in_features=2048, model='siamese1', ncriteria=10):
    if(model=='siamese1'):
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, 1))
    else:
        ret =  nn.Sequential(torch.nn.Linear(in_features, 256),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(256, 64),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(64, ncriteria))
    return ret
class ResNetSimCLR(nn.Module):

    def __init__(self, base_model, out_dim):
        super(ResNetSimCLR, self).__init__()
        self.resnet_dict = {"resnet18": models.resnet18(weights='ResNet18_Weights.DEFAULT', num_classes=out_dim),
                            "resnet50": models.resnet50(weights='ResNet50_Weights.DEFAULT', num_classes=out_dim),
                            "resnet101": models.resnet101(weights='ResNet101_Weights.DEFAULT', num_classes=out_dim),
                            "densenet121": models.densenet121(weights='DenseNet121_Weights.DEFAULT', num_classes=out_dim)}

        self.backbone = self._get_basemodel(base_model)
        dim_mlp = self.backbone.fc.in_features

        # add mlp projection head
        self.backbone.fc = nn.Sequential(nn.Linear(dim_mlp, dim_mlp), nn.ReLU(), self.backbone.fc)

    def _get_basemodel(self, model_name):
        try:
            model = self.resnet_dict[model_name]
        except KeyError:
            raise InvalidBackboneError(
                "Invalid backbone architecture. Check the config file and pass one of: resnet18 or resnet50")
        else:
            return model

    def forward(self, x):
        return self.backbone(x)

In [81]:
from torchvision import models, transforms

def get_feature_extractor(feature_extractor = 'resnet50', fcnet = None, cotrain=True, ncriteria=10, model='siamese1', simclr = None):
    if(feature_extractor == 'resnet50'):    
        fextractor = models.resnet50(weights='ResNet50_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet50', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'resnet101'):    
        fextractor = models.resnet101(weights='ResNet101_Weights.DEFAULT')
        in_features = 2048
        if(simclr):
            print('load simclr resnet')
            ressimclr = ResNetSimCLR('resnet101', 1000)
            state_dict = torch.load(simclr)
            ressimclr.load_state_dict(state_dict['state_dict'])
            fextractor = ressimclr.backbone
        fextractor.fc = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
    elif(feature_extractor == 'densnet121'):
        fextractor = models.densenet121(weights='DenseNet121_Weights.DEFAULT')
        in_features = 1024
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet
        # fextractor._modules['classifier'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vgg19'):
        fextractor = models.vgg19()
        fextractor.load_state_dict(torch.load('./pretrained/vgg19-dcbb9e9d.pth'))
        in_features = 25088 # https://www.geeksforgeeks.org/vgg-16-cnn-model/ length of vgg19
        fextractor.classifier = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor._modules['fc'] = fextractor._modules.pop('classifier')
    elif(feature_extractor == 'vit16'):
        fextractor = models.vit_b_16()
        in_features = 768
        fextractor.load_state_dict(torch.load('./pretrained/vit_b_16-c867db91.pth'))
        fextractor.heads.head = get_default_fc(in_features, model=model, ncriteria=ncriteria) if (fcnet == None) else fcnet 
        # fextractor.classifier = get_default_fc(in_features) if (fcnet == None) else fcnet
    else:
        assert False, 'No feature extractor founded'

    for param in fextractor.parameters():
            param.requires_grad = cotrain
    if(feature_extractor == 'resnet50' or feature_extractor == 'resnet101'):        
        for param in fextractor.fc.parameters():
            param.requires_grad = True
    elif(feature_extractor == 'vit16'):
        for param in fextractor.heads.parameters():
            param.requires_grad = True
    else:
        for param in fextractor.classifier.parameters():
            param.requires_grad = True

    return fextractor

In [82]:
class SiameseNetwork101(nn.Module):
    """
    Siamese neural network
    Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7
    Siamese ResNet-101 from Pytorch library
    """ 
    def __init__(self):
        super(SiameseNetwork101, self).__init__()
        # note that resnet101 requires 3 input channels, will repeat grayscale image x3
        self.cnn1 = get_feature_extractor(feature_extractor='resnet50', cotrain=False)# , simclr='/mnt/c/Users/PCM/Dropbox/pretrained/SimCLR/checkpoint_10_02102023.pth.tar')
        self.cnn1.fc = nn.Sequential(torch.nn.Linear(2048, 1000),
                                torch.nn.ReLU(),
                                torch.nn.Dropout(0.1),
                                torch.nn.Linear(1000, 256))
    
    def forward_once(self, x):
        output = self.cnn1(x)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

In [83]:
class SeverityModel(nn.Module):
    """
    Siamese neural network
    Modified from: https://hackernoon.com/facial-similarity-with-siamese-networks-in-pytorch-9642aa9db2f7
    Siamese ResNet-101 from Pytorch library
    """ 
    def __init__(self, path2pretrained=''):
        super(SeverityModel, self).__init__()
        # note that resnet101 requires 3 input channels, will repeat grayscale image x3
        self.bestsimese50simclr = SiameseNetwork101()
        if (path2pretrained):
            state_dict = torch.load(path2pretrained)
            self.bestsimese50simclr.load_state_dict(state_dict["model_state_dict"])
        self.bestsimese50simclr.cnn1.add_module('fc2',
            nn.Sequential(torch.nn.Linear(256, 256),
                          torch.nn.ReLU(),
                        torch.nn.Dropout(0.1),
                        torch.nn.Linear(256, 256)))
    
    def forward_once(self, x):
        output = self.bestsimese50simclr.cnn1.fc2(self.bestsimese50simclr.cnn1(x))
        return output

    def forward(self, input1, input2, refinput):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        refinput = self.bestsimese50simclr.cnn1(refinput)
        return output1, output2, refinput

# 3. Loss function

In [84]:
from torchvision.ops.focal_loss import sigmoid_focal_loss

def Focal_loss(class_logits,  labels):
    if class_logits.numel() == 0:
        return class_logits.new_zeros([1])[0]

    N = class_logits.shape[0]
    K = class_logits.shape[1] 

    target = class_logits.new_zeros(N, K)
    target[range(len(labels)), labels] = 1
    loss = sigmoid_focal_loss(class_logits, target, reduction = 'mean')
    return loss

# 4. Pipeline

In [86]:
for alpha in np.arange(0.3, 1, 0.1):
    config = {
        "data_path": "/mnt/d/AiThings/SimCLRxConPro/Dataset/papilledema",
        "batch_size": 8,
        "pretrain_encoder_checkpoint": f"/mnt/d/AiThings/SimCLRxConPro/upstream_task/papilledema/foundation model/new-proposal/lambda_{alpha}/best.pt",
        "num_epoch": 30,
        "checkpoint": "/mnt/d/AiThings/SimCLRxConPro/output/Papilledema/new-proposal",
        "repeat": 5
    }
    print(f"alpha: {alpha}")

    # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T02:49:52.921669Z","iopub.status.busy":"2024-08-01T02:49:52.921046Z","iopub.status.idle":"2024-08-01T02:49:53.288073Z","shell.execute_reply":"2024-08-01T02:49:53.286317Z"},"jupyter":{"outputs_hidden":false},"papermill":{"duration":0.377088,"end_time":"2024-08-01T02:49:53.290620","exception":false,"start_time":"2024-08-01T02:49:52.913532","status":"completed"},"tags":[]}
    image_datasets = {x: PapilledemaDataset(data_path = config["data_path"], phase=x,  seed =22) for x in ['train', 'val', 'test']}
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=config["batch_size"], shuffle=True, pin_memory = True)
                  for x in ['train', 'val', 'test']}

    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val',  'test']}
    class_names = ['1','2','3', '4', '5']

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(device, class_names)
    print(dataset_sizes)

    # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T02:49:53.308955Z","iopub.status.busy":"2024-08-01T02:49:53.308287Z","iopub.status.idle":"2024-08-01T02:49:56.680366Z","shell.execute_reply":"2024-08-01T02:49:56.679413Z"},"jupyter":{"outputs_hidden":false},"papermill":{"duration":3.383073,"end_time":"2024-08-01T02:49:56.682841","exception":false,"start_time":"2024-08-01T02:49:53.299768","status":"completed"},"tags":[]}
    import torch
    checkpoint = torch.load(config["pretrain_encoder_checkpoint"])

    # basemodel = SiameseNetwork101()
    # basemodel.load_state_dict(checkpoint["model_state_dict"])
    # classifierModel = basemodel.cnn1


    basemodel = SeverityModel()
    basemodel.load_state_dict(checkpoint["model_state_dict"])
    classifierModel = basemodel.bestsimese50simclr.cnn1
    del classifierModel.fc2

    classifierModel.fc = nn.Sequential(torch.nn.Linear(2048, 1000),
                                    torch.nn.ReLU(),
                                    torch.nn.Dropout(0.1),
                                    torch.nn.Linear(1000, 256),
                                    # torch.nn.Linear(256, 256),
                                    # torch.nn.ReLU(),
                                    # torch.nn.Dropout(0.1),
                                    # torch.nn.Linear(256, 256),
                                    torch.nn.ReLU(),
                                    torch.nn.Dropout(0.1),
                                    torch.nn.Linear(256, len(class_names)))

    default_cls_model = classifierModel

    # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T02:49:56.699873Z","iopub.status.busy":"2024-08-01T02:49:56.699576Z","iopub.status.idle":"2024-08-01T02:49:56.706645Z","shell.execute_reply":"2024-08-01T02:49:56.705792Z"},"jupyter":{"outputs_hidden":false},"papermill":{"duration":0.017673,"end_time":"2024-08-01T02:49:56.708615","exception":false,"start_time":"2024-08-01T02:49:56.690942","status":"completed"},"tags":[]}
    import torch.optim as optim
    from torch.optim import lr_scheduler

    momentum = 0.9
    lr = 1e-1
    optimizer_ft = optim.SGD([{'params': classifierModel.fc.parameters()}], lr=lr, momentum=momentum)
    loss_fn= Focal_loss
    scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=10, gamma=0.5)

    for param in classifierModel.parameters():
        param.requires_grad = False
    for param in classifierModel.fc.parameters():
        param.requires_grad = True

    # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T02:49:56.724664Z","iopub.status.busy":"2024-08-01T02:49:56.724397Z","iopub.status.idle":"2024-08-01T04:36:20.092023Z","shell.execute_reply":"2024-08-01T04:36:20.090957Z"},"jupyter":{"outputs_hidden":false},"papermill":{"duration":6383.508826,"end_time":"2024-08-01T04:36:20.224917","exception":false,"start_time":"2024-08-01T02:49:56.716091","status":"completed"},"tags":[]}
    from sklearn.metrics import f1_score
    from tqdm import tqdm

    # bestmodel = siamese50simclr
    for i in range(1, config["repeat"]+1):
        print("*"*100)
        print(f"Sample {i}")
        torch.cuda.empty_cache()
        classifierModel = default_cls_model.to(device)    
        f1max = 0
        for e in range(config["num_epoch"]):
            torch.cuda.empty_cache()
            training_acc = 0
            val_acc = 0
            training_loss_test = 0.0

            for inputs, labels in tqdm(dataloaders['train'], total= len(dataloaders['train'])):
                torch.cuda.empty_cache()
                classifierModel.train()
                inputs = inputs.to(device)
                labels = labels.to(device)
                # zero the parameter gradients
                optimizer_ft.zero_grad()

                outputs = classifierModel(inputs)
                _, preds = torch.max(outputs, 1)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer_ft.step()
                training_loss_test += loss.item() * inputs.size(0)
                training_acc += torch.sum(preds == labels.data)
            predlist = []
            labelist = []
            for inputs, labels in dataloaders['val']:
                classifierModel.eval()
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.no_grad():
                    outputs = classifierModel(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = loss_fn(outputs, labels)
                labelist.append(labels.detach().cpu().numpy()*1)
                predlist.append(preds.detach().cpu().numpy())
                val_acc += torch.sum(preds == labels.data)
            labelist = np.concatenate(labelist).ravel()
            predlist = np.concatenate(predlist).ravel()
            f1 = f1_score(predlist, labelist, average ='macro')
            if(f1 >= f1max):
                f1max = f1
                print(f"New best mode at epoch {e}")
                torch.save(classifierModel.state_dict(), os.path.join(config["checkpoint"], "best.pt"))
            torch.save(classifierModel.state_dict(), os.path.join(config["checkpoint"], "last.pt"))


            print(f"E{e} With LR {optimizer_ft.param_groups[0]['lr']} training acc: ", training_acc.detach().cpu().numpy() / dataset_sizes['training'], "Val acc: ", val_acc.detach().cpu().numpy() / dataset_sizes['valid'], "traning loss: ", training_loss_test / dataset_sizes['training'], "f1", f1)

        # %% [markdown] {"papermill":{"duration":0.192337,"end_time":"2024-08-01T04:36:20.554804","exception":false,"start_time":"2024-08-01T04:36:20.362467","status":"completed"},"tags":[]}
        # # 5. Evaluation

        # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T04:36:20.824083Z","iopub.status.busy":"2024-08-01T04:36:20.823606Z","iopub.status.idle":"2024-08-01T04:36:20.951305Z","shell.execute_reply":"2024-08-01T04:36:20.950473Z"},"papermill":{"duration":0.266946,"end_time":"2024-08-01T04:36:20.953394","exception":false,"start_time":"2024-08-01T04:36:20.686448","status":"completed"},"tags":[]}

        classifierModel.load_state_dict(torch.load(os.path.join(config["checkpoint"], "best.pt")))
        classifierModel = classifierModel.to(device)

        # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T04:36:21.217505Z","iopub.status.busy":"2024-08-01T04:36:21.216851Z","iopub.status.idle":"2024-08-01T04:38:42.198643Z","shell.execute_reply":"2024-08-01T04:38:42.197559Z"},"papermill":{"duration":141.115195,"end_time":"2024-08-01T04:38:42.201135","exception":false,"start_time":"2024-08-01T04:36:21.085940","status":"completed"},"tags":[]}
        test_acc = 0
        predlist = []
        labelist = []
        problist = []
        test_embeddings = torch.zeros((0, 2048))
        fextractor = torch.nn.Sequential(*(list(classifierModel.children())[:-1]))
        sedis = 0
        for inputs, labels in dataloaders['test']:
            classifierModel.eval()
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                outputs = classifierModel(inputs)
                emb = fextractor(inputs)
                _, preds = torch.max(outputs, 1)
                loss = loss_fn(outputs, labels)
                sedis = sedis + torch.sum(torch.exp(torch.abs(labels - torch.max(outputs, 1)[1])))
            problist.append(outputs[:,1].detach().cpu().numpy())
            labelist.append(labels.detach().cpu().numpy()*1)
            predlist.append(preds.detach().cpu().numpy())
            # test_embeddings  = torch.cat((test_embeddings, emb.detach().cpu().flatten().unsqueeze(0)), axis=0)
            test_acc += torch.sum(preds == labels.data)

        labelist = np.concatenate(labelist).ravel()
        problist = np.concatenate(problist).ravel()
        predlist = np.concatenate(predlist).ravel()
        # test_embeddings = np.array(test_embeddings)

        # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T04:38:42.473096Z","iopub.status.busy":"2024-08-01T04:38:42.472443Z","iopub.status.idle":"2024-08-01T04:38:42.553219Z","shell.execute_reply":"2024-08-01T04:38:42.552298Z"},"papermill":{"duration":0.22059,"end_time":"2024-08-01T04:38:42.555147","exception":false,"start_time":"2024-08-01T04:38:42.334557","status":"completed"},"tags":[]}
        print(sedis/dataset_sizes['test'])

        # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T04:38:42.816694Z","iopub.status.busy":"2024-08-01T04:38:42.816372Z","iopub.status.idle":"2024-08-01T04:38:42.822368Z","shell.execute_reply":"2024-08-01T04:38:42.821446Z"},"papermill":{"duration":0.138653,"end_time":"2024-08-01T04:38:42.824209","exception":false,"start_time":"2024-08-01T04:38:42.685556","status":"completed"},"tags":[]}
        print("test_acc acc: ", test_acc / dataset_sizes['test'])


        # %% [code] {"execution":{"iopub.execute_input":"2024-08-01T04:38:43.087250Z","iopub.status.busy":"2024-08-01T04:38:43.086594Z","iopub.status.idle":"2024-08-01T04:38:43.106226Z","shell.execute_reply":"2024-08-01T04:38:43.105087Z"},"papermill":{"duration":0.153887,"end_time":"2024-08-01T04:38:43.108296","exception":false,"start_time":"2024-08-01T04:38:42.954409","status":"completed"},"tags":[]}
        from sklearn.metrics import classification_report
        from sklearn.metrics import roc_auc_score

        print(classification_report(labelist, predlist, digits=3))

alpha: 0.3
cuda:0 ['1', '2', '3', '4', '5']
{'train': 957, 'val': 204, 'test': 208}


/tmp/ipykernel_2262115/3806444990.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(config["pretrain_encoder_checkpoint"])


****************************************************************************************************
Sample 1


100%|██████████| 120/120 [00:10<00:00, 11.86it/s]


New best mode at epoch 0


KeyError: 'training'

In [ ]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# cm = confusion_matrix(labelist, predlist)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
# disp.plot()
# plt.savefig("/kaggle/working/confusion_matrix.png")
# plt.show()